# 02 — Brand extraction

Extract brand from ITEMDESC on sales and chargebacks using prefix map (auto + manual) with a keyword fallback. Verify coverage is high enough to trust downstream promo joins.

**Upstream:** `sales.parquet`, `cb.parquet`, `tpr.parquet` from `01_load.ipynb`.

**Outputs:**
- `sales_with_brand.parquet` — sales + new `brand` column
- `tpr_with_brand.parquet` — chargeback TPR subset + `brand` column
- `sku_brand.parquet` — compact `ITEMNMBR -> brand` lookup

**Promotes to:** `src/brand.py` once coverage + SKU-brand consistency check pass.

## 1. Imports

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'pipeline' else Path.cwd()
ART = ROOT / 'pipeline' / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)

# Make `src/` importable from this notebook
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.load import load_cached

## 2. Load upstream

In [2]:
dfs = load_cached()
sales = dfs['sales']
cb    = dfs['cb']
tpr   = dfs['tpr'].copy()  # will add columns

print(f'sales: {sales.shape}   cb: {cb.shape}   tpr: {tpr.shape}')

sales: (236818, 23)   cb: (18804, 13)   tpr: (6868, 13)


## 3. Do the work

In [3]:
# ---- Step A: substring extractor (v1) ---------------------------------------
# Seed brand list. Order matters only where one term is a substring of another;
# the loop returns on first match.
BRANDS = [
    'tiger balm', 'ginger chew', 'ferrero', 'ricola', 'kwan loong',
    'ginseng', 'kjeldsens', 'nutella', 'totole', 'bee & flower',
    'am gsg', 'pop ginger',
]


def extract_brand(desc):
    """Substring-match a description against the seed brand list. Returns None if no hit."""
    if pd.isna(desc):
        return None
    d = str(desc).lower()
    for b in BRANDS:
        if b in d:
            return b
    return None


def first_token(s):
    """Leading alphabetic token (e.g. 'TBR' from 'TBR-1234 ...'). Used as a prefix key."""
    if pd.isna(s):
        return None
    m = re.match(r'([A-Za-z&]+)', str(s))
    return m.group(1).upper() if m else None


tpr['brand'] = tpr['Item Description'].apply(extract_brand)
matched  = tpr[tpr['brand'].notna()].copy()
no_brand = tpr[tpr['brand'].isna()].copy()
matched['prefix']  = matched['Item Description'].apply(first_token)
no_brand['prefix'] = no_brand['Item Description'].apply(first_token)

print(f'v1 substring coverage : {tpr["brand"].notna().mean()*100:.1f}% of tpr rows')
print(f'  matched rows : {len(matched):,}')
print(f'  unmatched    : {len(no_brand):,}')

# ---- Step B: auto-derive PREFIX_MAP_AUTO from matched rows -------------------
# For each prefix, pick the dominant brand IF share >= 0.9 AND support >= 3.
prefix_brand = (
    matched.groupby(['prefix', 'brand']).size().rename('n').reset_index()
)
prefix_totals = prefix_brand.groupby('prefix')['n'].sum().rename('total')
prefix_brand = prefix_brand.merge(prefix_totals, on='prefix')
prefix_brand['share'] = prefix_brand['n'] / prefix_brand['total']

trust = (
    prefix_brand.sort_values('n', ascending=False)
                .drop_duplicates('prefix')
)
trust = trust[(trust['share'] >= 0.9) & (trust['n'] >= 3)]
PREFIX_MAP_AUTO = dict(zip(trust['prefix'], trust['brand']))

print(f'\nAuto-derived prefix map ({len(PREFIX_MAP_AUTO)} entries, share>=0.9, n>=3):')
for p, b in sorted(PREFIX_MAP_AUTO.items()):
    n = int(trust.set_index('prefix').loc[p, 'n'])
    print(f'  {p:<6s} -> {b:<20s}  (support={n})')

# ---- Step C: PREFIX_MAP_MANUAL (hand-labeled from diagnostic inspection) ----
# These keys are first-alphabetic-tokens of the *Item Description* (chargeback-style IDs).
# Do NOT put SKU-code prefixes here (SKU codes live in ITEMNMBR, not ITEMDESC).
PREFIX_MAP_MANUAL = {
    'GHC': 'ginger honey crystals',  # new brand
    'BFS': 'bee & flower',           # B&F Soap
    'CFX': 'cofixrx',                # new brand
    'AZX': 'azzurx',                 # new brand
    'KGS': 'ginseng',                # Korean Ginseng
    'TBR': 'tiger balm',
    'TN':  'tiger balm',
    'TA':  'tiger balm',
    'T':   'tiger balm',
    'HAN': 'hans honey',             # Hans Honey Loquat Candy
}
# Intentionally NOT mapped (admin / promo-meta, not brand):
#   MIS, LOA, LOC, PRICE, TRADE, SELLER, RETAILER, APRIL, POP, SALES

PREFIX_MAP = {**PREFIX_MAP_AUTO, **PREFIX_MAP_MANUAL}
BRANDS_V2  = BRANDS + ['ginger honey crystals', 'cofixrx', 'azzurx', 'hans honey', 'mx eggrolls']

# SKU-code prefix → brand overrides (keyed on ITEMNMBR, applied to sales only)
SKU_PREFIX_MAP = {
    'D-': 'pop tea',  # all D-* SKUs are POP-branded teas (green/jasmine/oolong/etc.)
}

# ---- Step D: diagnostic — prefixes in unmatched rows the COMBINED map still misses
no_brand_prefix_counts = no_brand['prefix'].value_counts()
still_unknown = no_brand_prefix_counts[~no_brand_prefix_counts.index.isin(PREFIX_MAP)]
print(f'\nPrefixes still unmapped after AUTO+MANUAL ({len(still_unknown)} unique). Top 20:')
for prefix in still_unknown.head(20).index:
    sample = no_brand[no_brand['prefix'] == prefix]['Item Description'].iloc[0]
    n = int(still_unknown[prefix])
    print(f'  {prefix:<6s} (n={n:>4})  e.g. "{str(sample)[:70]}"')


def extract_brand_v2(desc):
    """Prefix-map first (chargeback-style IDs), then substring fallback (sales descriptions)."""
    if pd.isna(desc):
        return None
    s = str(desc)
    prefix = first_token(s)
    if prefix and prefix in PREFIX_MAP:
        return PREFIX_MAP[prefix]
    d = s.lower()
    for b in BRANDS_V2:
        if b in d:
            return b
    return None


def apply_sku_prefix_override(df, sku_col='ITEMNMBR', brand_col='brand'):
    """Fill null brand from SKU_PREFIX_MAP based on ITEMNMBR (sales-only: D-* -> pop tea)."""
    if brand_col not in df.columns:
        df[brand_col] = pd.NA
    for sku_prefix, brand in SKU_PREFIX_MAP.items():
        mask = df[brand_col].isna() & df[sku_col].astype(str).str.startswith(sku_prefix)
        df.loc[mask, brand_col] = brand
    return df


def fill_brand_from_sku_majority(df, sku_col='ITEMNMBR', brand_col='brand'):
    """Last-resort fill: for null-brand rows, inherit brand from other rows with same SKU.
    Handles edge cases like a single row with a null ITEMDESC whose SKU is otherwise labeled."""
    sku_to_brand = (
        df.dropna(subset=[brand_col])
          .groupby(sku_col)[brand_col]
          .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)
    )
    mask = df[brand_col].isna()
    df.loc[mask, brand_col] = df.loc[mask, sku_col].map(sku_to_brand)
    return df


# Apply v2 to both sources, then SKU-prefix + SKU-majority fills on sales
tpr['brand_v2']    = tpr['Item Description'].apply(extract_brand_v2)
sales_brand        = sales['ITEMDESC'].apply(extract_brand_v2)
sales = sales.assign(brand=sales_brand)
sales = apply_sku_prefix_override(sales)
sales = fill_brand_from_sku_majority(sales)

print(f'\nFinal brand coverage (v2 + SKU override + SKU majority):')
print(f'  tpr   : {tpr["brand_v2"].notna().mean()*100:5.1f}%   ({tpr["brand_v2"].notna().sum():,} / {len(tpr):,})')
print(f'  sales : {sales["brand"].notna().mean()*100:5.1f}%   ({sales["brand"].notna().sum():,} / {len(sales):,})')

v1 substring coverage : 25.1% of tpr rows
  matched rows : 1,721
  unmatched    : 5,147

Auto-derived prefix map (16 entries, share>=0.9, n>=3):
  AGS    -> am gsg                (support=54)
  ASG    -> am gsg                (support=5)
  CGZ    -> ginger chew           (support=10)
  GCP    -> ginger chew           (support=473)
  GCZ    -> ginger chew           (support=465)
  GSE    -> ginseng               (support=23)
  GSG    -> ginseng               (support=34)
  TB     -> tiger balm            (support=10)
  TBP    -> tiger balm            (support=5)
  TBU    -> tiger balm            (support=4)
  TBZ    -> tiger balm            (support=352)
  TEA    -> am gsg                (support=44)
  TM     -> tiger balm            (support=23)
  TP     -> tiger balm            (support=49)
  TU     -> tiger balm            (support=110)
  TZ     -> tiger balm            (support=23)

Prefixes still unmapped after AUTO+MANUAL (4 unique). Top 20:
  MIS    (n= 725)  e.g. "MIS - 02/24  F

## 4. Validate

In [4]:
# ---- Coverage by source -----------------------------------------------------
print('TPR brand counts (v2):')
print(tpr['brand_v2'].value_counts(dropna=False).to_string())

print('\nSales brand counts (v2):')
print(sales['brand'].value_counts(dropna=False).to_string())

# ---- SKU -> brand consistency: each ITEMNMBR should map to exactly one brand
sku_brand = (
    sales.dropna(subset=['brand'])
         .groupby('ITEMNMBR')['brand']
         .agg(lambda s: set(s))
)
multi = sku_brand[sku_brand.map(len) > 1]
print(f'\nSKUs with >1 brand label: {len(multi)}')
if len(multi):
    print(multi.head(10).to_string())

# ---- Unmatched sales SKUs (rows where brand is null) ------------------------
unmatched_skus = sales[sales['brand'].isna()]['ITEMNMBR'].value_counts()
print(f'\nSales SKUs with no brand ({len(unmatched_skus)} unique SKUs, '
      f'{sales["brand"].isna().sum():,} rows):')
print(unmatched_skus.head(15).to_string())

# Show one sample ITEMDESC per unmatched SKU so we can judge whether BRANDS needs extending
print('\nSample ITEMDESC per top unmatched SKU:')
for sku in unmatched_skus.head(10).index:
    desc = sales[sales['ITEMNMBR'] == sku]['ITEMDESC'].dropna().iloc[0]
    print(f'  {sku:<12s}  "{str(desc)[:70]}"')

TPR brand counts (v2):
brand_v2
tiger balm               4038
ginger chew              1350
NaN                       849
am gsg                    252
ginger honey crystals     125
bee & flower               89
ginseng                    77
cofixrx                    61
hans honey                 17
azzurx                     10

Sales brand counts (v2):
brand
tiger balm     73489
ginger chew    67531
pop tea        33134
am gsg         30984
ferrero        18371
totole          4350
kjeldsens       4252
kwan loong      4112
mx eggrolls      595

SKUs with >1 brand label: 0

Sales SKUs with no brand (0 unique SKUs, 0 rows):
Series([], )

Sample ITEMDESC per top unmatched SKU:


## 5. Save downstream artifact

In [5]:
# Normalize column name: tpr uses 'brand_v2' internally to compare vs v1; write it as 'brand'.
tpr_out = tpr.drop(columns=['brand']).rename(columns={'brand_v2': 'brand'})

# Per-SKU brand lookup (handy for downstream joins on ITEMNMBR)
sku_brand_map = (
    sales.dropna(subset=['brand'])
         .drop_duplicates('ITEMNMBR')
         [['ITEMNMBR', 'brand']]
         .reset_index(drop=True)
)

tpr_out.to_parquet(ART / 'tpr_with_brand.parquet')
sales.to_parquet(ART / 'sales_with_brand.parquet')
sku_brand_map.to_parquet(ART / 'sku_brand.parquet')

print(f'tpr_with_brand    {tpr_out.shape}  -> tpr_with_brand.parquet')
print(f'sales_with_brand  {sales.shape}    -> sales_with_brand.parquet')
print(f'sku_brand         {sku_brand_map.shape}    -> sku_brand.parquet')

tpr_with_brand    (6868, 14)  -> tpr_with_brand.parquet
sales_with_brand  (236818, 24)    -> sales_with_brand.parquet
sku_brand         (83, 2)    -> sku_brand.parquet


## 6. Promote

Once validation above looks right, extract the core logic into `src/brand.py` and replace the inline code here with `from src.<module> import ...`. Downstream dev notebooks can then import the same function.